In [5]:
%pip install pandas numpy matplotlib pyarrow

Note: you may need to restart the kernel to use updated packages.


In [4]:
import sqlite3
import pandas as pd
import numpy as np
import os
import logging
from pathlib import Path
from datetime import datetime

# ── Logging setup ──────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%H:%M:%S'
)
logger = logging.getLogger('etl')

# ── File paths ─────────────────────────────────────────────────────────────────
# Changed DATA_DIR to /content to match your file structure
DATA_DIR = Path('.')
DB_PATH       = DATA_DIR / 'PerfectPetClub.db'
CRM_PATH      = DATA_DIR / 'PerfectPetClub_CRM_report.csv'
LOG_PATH      = DATA_DIR / 'PerfectPetClub_server_log.csv'
SURVEY_PATH   = DATA_DIR / 'PerfectPetClub_survey (2).csv'
OUTPUT_PARQUET = DATA_DIR / 'churn_dataset.parquet'

# Reference date for recency calculations (last day with sales in the dataset)
REFERENCE_DATE = pd.Timestamp('2026-11-10')

logger.info('Configuration loaded. Reference date: %s', REFERENCE_DATE.date())

19:42:45 [INFO] Configuration loaded. Reference date: 2026-11-10


In [5]:
def extract_db(db_path: Path) -> dict:
    """
    Extract all relevant tables from the PerfectPetClub SQLite database.

    Parameters
    ----------
    db_path : Path
        File-system path to the .db file.

    Returns
    -------
    dict
        Dictionary of {table_name: DataFrame} for each extracted table.
    """
    logger.info('EXTRACT: connecting to SQLite database at %s', db_path)
    conn = sqlite3.connect(db_path)

    tables = {
        'customer':          'SELECT * FROM customer',
        'subscription':      'SELECT * FROM subscription',
        'subscription_type': 'SELECT * FROM subscription_type',
        'sale':              'SELECT * FROM sale',
        'payment':           'SELECT * FROM payment',
        'sale_product':      'SELECT * FROM sale_product',
        'product':           'SELECT * FROM product',
        'product_category':  'SELECT * FROM product_category',
        'product_review':    'SELECT * FROM product_review',
    }

    raw = {}
    for name, query in tables.items():
        raw[name] = pd.read_sql_query(query, conn)
        logger.info('  Extracted table %-20s shape: %s', name, raw[name].shape)

    conn.close()
    logger.info('EXTRACT DB: complete – %d tables loaded', len(raw))
    return raw


def extract_crm(crm_path: Path) -> pd.DataFrame:
    """
    Extract the CRM customer service contact report from CSV.

    Parameters
    ----------
    crm_path : Path
        Path to PerfectPetClub_CRM_report.csv

    Returns
    -------
    pd.DataFrame
        Raw CRM DataFrame.
    """
    logger.info('EXTRACT: reading CRM report from %s', crm_path)
    df = pd.read_csv(crm_path)
    logger.info('EXTRACT CRM: shape %s', df.shape)
    return df


def extract_server_log(log_path: Path) -> pd.DataFrame:
    """
    Extract the web application server login log from CSV.

    Parameters
    ----------
    log_path : Path
        Path to PerfectPetClub_server_log.csv

    Returns
    -------
    pd.DataFrame
        Raw server log DataFrame.
    """
    logger.info('EXTRACT: reading server log from %s', log_path)
    df = pd.read_csv(log_path)
    logger.info('EXTRACT SERVER LOG: shape %s', df.shape)
    return df


def extract_survey(survey_path: Path) -> pd.DataFrame:
    """
    Extract customer survey responses from CSV.
    If the file is not found, returns an empty DataFrame with expected columns
    so that downstream transformations remain robust.

    Parameters
    ----------
    survey_path : Path
        Path to PerfectPetClub_survey.csv

    Returns
    -------
    pd.DataFrame
        Raw survey DataFrame, or empty DataFrame if file unavailable.
    """
    if not survey_path.exists():
        logger.warning('Survey file not found at %s – returning empty DataFrame', survey_path)
        return pd.DataFrame(columns=['email', 'reported_income',
                                     'nps_recommend', 'satisfaction_score',
                                     'online_comparison_freq'])
    logger.info('EXTRACT: reading survey from %s', survey_path)
    df = pd.read_csv(survey_path)
    logger.info('EXTRACT SURVEY: shape %s', df.shape)
    return df

In [6]:
# ── Run all extractions ────────────────────────────────────────────────────────
db_tables  = extract_db(DB_PATH)
raw_crm    = extract_crm(CRM_PATH)
raw_log    = extract_server_log(LOG_PATH)
raw_survey = extract_survey(SURVEY_PATH)

# Quick sanity check – print head of each raw source
print('\n=== Customer (DB) ===')
display(db_tables['customer'].head(3))

print('\n=== Subscription (DB) ===')
display(db_tables['subscription'].head(3))

print('\n=== CRM ===')
display(raw_crm.head(3))

print('\n=== Server Log ===')
display(raw_log.head(3))

19:42:45 [INFO] EXTRACT: connecting to SQLite database at PerfectPetClub.db
19:42:45 [INFO]   Extracted table customer             shape: (221, 9)
19:42:45 [INFO]   Extracted table subscription         shape: (221, 5)
19:42:45 [INFO]   Extracted table subscription_type    shape: (3, 3)
19:42:45 [INFO]   Extracted table sale                 shape: (23188, 6)
19:42:45 [INFO]   Extracted table payment              shape: (23188, 4)
19:42:45 [INFO]   Extracted table sale_product         shape: (59759, 4)
19:42:45 [INFO]   Extracted table product              shape: (194, 4)
19:42:45 [INFO]   Extracted table product_category     shape: (26, 2)
19:42:46 [INFO]   Extracted table product_review       shape: (10990, 5)
19:42:46 [INFO] EXTRACT DB: complete – 9 tables loaded
19:42:46 [INFO] EXTRACT: reading CRM report from PerfectPetClub_CRM_report.csv
19:42:46 [INFO] EXTRACT CRM: shape (7585, 4)
19:42:46 [INFO] EXTRACT: reading server log from PerfectPetClub_server_log.csv
19:42:46 [INFO] EXTRAC


=== Customer (DB) ===


,id,name,email,password_hash,address_line_1,address_line_2,county,postcode,subscription_id
0,1,Anahit Sargsyan,Sargsyan701@somemail.co.uk,6741fa5591efb6a5310f241a22ca305eb0449a4bb3dd20...,76 High Street,Maidenhead,Berkshire,SL6 1CD,1
1,2,Leyla Mammadova,Mammadova282@example.com,3931efc8f036b658429849e8b951f462f750d1c6b15536...,18 High Street,Chipping Norton,Oxfordshire,OX7 5GH,2
2,3,Sana Shah,Sana858@notrealemail.com,db0af1e14f9d10b440a960e2de9ada9ad83d92449189f1...,77 High Street,Kidderminster,Worcestershire,DY10 2JK,3



=== Subscription (DB) ===


,id,start_date,end_date,auto_renew,subscription_type_id
0,1,2026-01-01 00:00:00.000,2026-01-31 00:00:00.000,0,3
1,2,2026-01-01 00:00:00.000,2026-11-27 00:00:00.000,1,3
2,3,2026-01-01 00:00:00.000,2026-11-27 00:00:00.000,1,1



=== CRM ===


,customer_id,recieved_datatime,platform,type
0,55,2026-01-01 00:42:47.000,['e-mail'],complaint
1,49,2026-01-01 01:25:34.000,['live-chat'],query
2,44,2026-01-01 03:33:55.001,['telephone'],query



=== Server Log ===


,datetime,session_id,ip,login,user_agent
0,2026-01-01 02:08:21.000,session_cookie::63fec84dedaa4f2cbaa10b413b5a7d...,239.66.231.182,Sipho928@somemail.co.uk,"""Mozilla/5.0 (Desktop/Laptop; Cipher CoreBook ..."
1,2026-01-01 02:51:08.001,session_cookie::64bcd32ed1494bd1af6b7a233cd638...,200.200.140.49,Sienna.Martin923@notrealemail.com,"""Mozilla/5.0 (Mobile; Pulse PulsePhone) AppleW..."
2,2026-01-01 04:16:42.001,session_cookie::f848746c9c96422b9eb050eccacfeb...,38.219.218.103,v134@somemail.co.uk,"""Mozilla/5.0 (Mobile; Titan Pocket Go) AppleWe..."


In [7]:
### transform stage

In [8]:
def check_quality(df: pd.DataFrame, name: str, required_cols: list = None) -> pd.DataFrame:
    """
    Run a standard set of data quality checks on a DataFrame and log results.

    Checks performed:
    - Missing values per column
    - Duplicate rows
    - Column presence (if required_cols supplied)
    - Basic row/col counts

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame to validate.
    name : str
        Human-readable label used in log messages.
    required_cols : list, optional
        List of column names that must be present.

    Returns
    -------
    pd.DataFrame
        The original DataFrame (unchanged). Quality results are logged only.
    """
    logger.info('QUALITY CHECK: %s  rows=%d  cols=%d', name, *df.shape)

    # Missing values
    null_counts = df.isnull().sum()
    null_cols = null_counts[null_counts > 0]
    if null_cols.empty:
        logger.info('  [OK] No missing values')
    else:
        for col, cnt in null_cols.items():
            pct = cnt / len(df) * 100
            logger.warning('  [WARN] Missing: %s = %d (%.1f%%)', col, cnt, pct)

    # Duplicate rows
    n_dupes = df.duplicated().sum()
    if n_dupes == 0:
        logger.info('  [OK] No duplicate rows')
    else:
        logger.warning('  [WARN] %d duplicate rows found', n_dupes)

    # Required columns
    if required_cols:
        missing_cols = [c for c in required_cols if c not in df.columns]
        if missing_cols:
            logger.error('  [ERROR] Missing required columns: %s', missing_cols)
        else:
            logger.info('  [OK] All required columns present')

    return df

In [9]:
def transform_db(raw: dict, reference_date: pd.Timestamp) -> pd.DataFrame:
    """
    Transform SQLite database tables into a customer-level feature DataFrame.
    """
    logger.info('TRANSFORM DB: starting')

    # ── Quality checks ─────────────────────────────────────────────────────────
    check_quality(raw['customer'],     'customer',     ['id', 'email', 'subscription_id'])
    check_quality(raw['subscription'], 'subscription', ['id', 'auto_renew', 'subscription_type_id'])
    check_quality(raw['sale'],         'sale',         ['id', 'customer_id', 'payment_id'])
    check_quality(raw['payment'],      'payment',      ['id', 'amount'])

    # ── Parse date columns ─────────────────────────────────────────────────────
    sub = raw['subscription'].copy()
    sub['start_date']  = pd.to_datetime(sub['start_date'],  errors='coerce')
    sub['end_date']    = pd.to_datetime(sub['end_date'],    errors='coerce')

    sale = raw['sale'].copy()
    sale['order_date']     = pd.to_datetime(sale['order_date'],     errors='coerce')
    sale['expected_date']  = pd.to_datetime(sale['expected_date'],  errors='coerce')
    sale['delivered_date'] = pd.to_datetime(sale['delivered_date'], errors='coerce')

    # ── Validate auto_renew values ─────────────────────────────────────────────
    invalid_renew = sub[~sub['auto_renew'].isin([0, 1])]
    if not invalid_renew.empty:
        logger.warning('  [WARN] %d rows with invalid auto_renew values', len(invalid_renew))
    else:
        logger.info('  [OK] auto_renew values all valid (0 or 1)')

    # ── Validate payments ──────────────────────────────────────────────────────
    pay = raw['payment'].copy()
    neg_payments = pay[pay['amount'] <= 0]
    if not neg_payments.empty:
        logger.warning('  [WARN] %d payment rows with non-positive amounts', len(neg_payments))
    else:
        logger.info('  [OK] All payment amounts positive')

    # ── Base: customer + subscription ──────────────────────────────────────────
    cust = raw['customer'].copy()
    sub_type = raw['subscription_type'].rename(columns={'id': 'subscription_type_id',
                                                         'name': 'subscription_type_name'})
    sub = sub.merge(sub_type[['subscription_type_id', 'subscription_type_name']],
                    on='subscription_type_id', how='left')

    sub['subscription_days'] = (sub['end_date'] - sub['start_date']).dt.days
    sub['churn'] = (sub['auto_renew'] == 0).astype(int)

    # FIXED: Added 'subscription_id' to the sliced list of columns so it is available for merging
    base = cust[['id', 'email', 'subscription_id']].merge(
        sub[['id', 'subscription_type_name', 'subscription_type_id',
             'subscription_days', 'auto_renew', 'churn']],
        left_on='subscription_id', right_on='id',
        how='left',
        suffixes=('', '_sub')
    ).rename(columns={'id': 'customer_id'})

    if 'id_sub' in base.columns:
        base = base.drop(columns=['id_sub'])

    # ── Order / payment features ────────────────────────────────────────────────
    sale_pay = sale.merge(pay[['id', 'amount']],
                          left_on='payment_id', right_on='id', how='left')

    order_feats = sale_pay.groupby('customer_id').agg(
        num_orders          = ('id_x', 'count'),
        total_spend_pence   = ('amount', 'sum'),
        days_since_last_order = ('order_date',
                                  lambda x: (reference_date - x.max()).days)
    ).reset_index()
    order_feats['avg_order_value'] = (
        order_feats['total_spend_pence'] / order_feats['num_orders']
    ).round(2)

    # ── Product variety ─────────────────────────────────────────────────────────
    sp = raw['sale_product'].copy()
    sp_sale = sp.merge(sale[['id', 'customer_id']], left_on='sale_id', right_on='id', how='left')
    prod_variety = sp_sale.groupby('customer_id')['product_id'].nunique().reset_index()
    prod_variety.columns = ['customer_id', 'num_distinct_products']

    # ── Review features ─────────────────────────────────────────────────────────
    review_feats = raw['product_review'].groupby('customer_id').agg(
        num_reviews      = ('id', 'count'),
        avg_review_rating = ('star_rating', 'mean')
    ).reset_index()
    review_feats['avg_review_rating'] = review_feats['avg_review_rating'].round(2)

    # ── Delivery delay ──────────────────────────────────────────────────────────
    delivered = sale.dropna(subset=['expected_date', 'delivered_date']).copy()
    delivered['delivery_delay_days'] = (
        (delivered['delivered_date'] - delivered['expected_date']).dt.days
    )
    delay_feats = delivered.groupby('customer_id')['delivery_delay_days'].mean().reset_index()
    delay_feats.columns = ['customer_id', 'avg_delivery_delay_days']
    delay_feats['avg_delivery_delay_days'] = delay_feats['avg_delivery_delay_days'].round(2)

    # ── Merge all DB features ───────────────────────────────────────────────────
    df = base.copy()
    for feats in [order_feats, prod_variety, review_feats, delay_feats]:
        df = df.merge(feats, on='customer_id', how='left')

    count_cols = ['num_orders', 'total_spend_pence', 'avg_order_value',
                  'num_distinct_products', 'num_reviews', 'avg_review_rating']
    df[count_cols] = df[count_cols].fillna(0)

    logger.info('TRANSFORM DB: complete  shape=%s  churn_rate=%.1f%%',
                df.shape, df['churn'].mean() * 100)
    return df

In [10]:
def transform_crm(raw_crm: pd.DataFrame, reference_date: pd.Timestamp) -> pd.DataFrame:
    """
    Transform raw CRM contact data into customer-level behavioural features.

    Feature engineering performed:
    - num_contacts            : total CRM interactions
    - num_complaints          : interactions flagged as 'complaint'
    - num_queries             : interactions flagged as 'query'
    - complaint_ratio         : proportion of contacts that are complaints
    - days_since_last_contact : recency of most recent CRM interaction
    - crm_platforms           : set of channels used (email/telephone/live-chat)

    Data quality performed:
    - Parse datetime column (typo 'recieved_datatime')
    - Validate type values are in expected set
    - Remove rows with null customer_id

    Parameters
    ----------
    raw_crm : pd.DataFrame
        Raw CRM DataFrame from extract_crm().
    reference_date : pd.Timestamp
        Date used as 'today' for recency calculation.

    Returns
    -------
    pd.DataFrame
        One row per customer with CRM-derived features.
    """
    logger.info('TRANSFORM CRM: starting  raw_shape=%s', raw_crm.shape)

    df = raw_crm.copy()
    check_quality(df, 'CRM', ['customer_id', 'type'])

    # ── Fix known column-name typo and parse datetime ──────────────────────────
    df = df.rename(columns={'recieved_datatime': 'contact_datetime'})
    df['contact_datetime'] = pd.to_datetime(df['contact_datetime'], errors='coerce')
    n_bad_dt = df['contact_datetime'].isna().sum()
    if n_bad_dt:
        logger.warning('  [WARN] %d CRM rows with unparseable datetime – dropping', n_bad_dt)
        df = df.dropna(subset=['contact_datetime'])

    # ── Validate type column ───────────────────────────────────────────────────
    expected_types = {'complaint', 'query'}
    actual_types   = set(df['type'].str.strip().str.lower().unique())
    unexpected     = actual_types - expected_types
    if unexpected:
        logger.warning('  [WARN] Unexpected CRM type values: %s', unexpected)
    else:
        logger.info('  [OK] CRM type values valid: %s', actual_types)
    df['type'] = df['type'].str.strip().str.lower()

    # ── Drop missing customer_id ───────────────────────────────────────────────
    before = len(df)
    df = df.dropna(subset=['customer_id'])
    df['customer_id'] = df['customer_id'].astype(int)
    logger.info('  Dropped %d rows with null customer_id', before - len(df))

    # ── Feature engineering ────────────────────────────────────────────────────
    agg = df.groupby('customer_id').agg(
        num_contacts          = ('type', 'count'),
        num_complaints        = ('type', lambda x: (x == 'complaint').sum()),
        num_queries           = ('type', lambda x: (x == 'query').sum()),
        days_since_last_contact = ('contact_datetime',
                                    lambda x: (reference_date - x.max()).days)
    ).reset_index()

    agg['complaint_ratio'] = (
        agg['num_complaints'] / agg['num_contacts'].replace(0, np.nan)
    ).round(3)

    # ── Platform usage (one-hot for each channel) ──────────────────────────────
    # platform column is stored as a string representation of a list e.g. "['e-mail']"
    df['platform_clean'] = df['platform'].str.strip("[']")
    for channel in ['e-mail', 'telephone', 'live-chat']:
        col_name = 'uses_' + channel.replace('-', '_')
        flag = df.groupby('customer_id')['platform_clean'].apply(
            lambda x: int(x.str.contains(channel, na=False).any())
        ).reset_index(name=col_name)
        agg = agg.merge(flag, on='customer_id', how='left')

    logger.info('TRANSFORM CRM: complete  shape=%s', agg.shape)
    return agg

In [11]:
def transform_server_log(raw_log: pd.DataFrame,
                          customer_emails: pd.Series,
                          reference_date: pd.Timestamp) -> pd.DataFrame:
    """
    Transform server login log into customer-level engagement features.
    Login rows are matched to customers via email address.

    Feature engineering performed:
    - num_logins              : total login events
    - login_frequency_per_day : num_logins / (date range of logins)
    - days_since_last_login   : recency of most recent login
    - device_type_mobile      : 1 if ever logged in from a mobile device
    - device_type_tablet      : 1 if ever logged in from a tablet
    - device_type_desktop     : 1 if ever logged in from a desktop/laptop
    - num_unique_ips          : count of distinct IP addresses used

    Parameters
    ----------
    raw_log : pd.DataFrame
        Raw server log DataFrame from extract_server_log().
    customer_emails : pd.Series
        Series mapping customer_id -> email, used to join log to customers.
    reference_date : pd.Timestamp
        Date used as 'today' for recency calculation.

    Returns
    -------
    pd.DataFrame
        One row per customer (matched by email) with log-derived features.
    """
    logger.info('TRANSFORM LOG: starting  raw_shape=%s', raw_log.shape)

    df = raw_log.copy()
    check_quality(df, 'server_log', ['datetime', 'login', 'ip'])

    # ── Parse datetime ─────────────────────────────────────────────────────────
    df['login_dt'] = pd.to_datetime(df['datetime'], errors='coerce')
    n_bad = df['login_dt'].isna().sum()
    if n_bad:
        logger.warning('  [WARN] %d log rows with unparseable datetime – dropping', n_bad)
        df = df.dropna(subset=['login_dt'])

    # ── Normalise email ────────────────────────────────────────────────────────
    df['login'] = df['login'].str.strip().str.lower()

    # ── Map email → customer_id ────────────────────────────────────────────────
    # customer_emails is a Series: index=customer_id, values=email
    email_to_id = customer_emails.str.strip().str.lower().reset_index()
    email_to_id.columns = ['customer_id', 'login']

    df = df.merge(email_to_id, on='login', how='left')
    unmatched = df['customer_id'].isna().sum()
    if unmatched:
        logger.warning('  [WARN] %d log entries not matched to a customer', unmatched)
    df = df.dropna(subset=['customer_id'])
    df['customer_id'] = df['customer_id'].astype(int)

    # ── Extract device type from user_agent ────────────────────────────────────
    ua = df['user_agent'].fillna('')
    df['is_mobile']  = ua.str.contains('Mobile',          case=False, na=False).astype(int)
    df['is_tablet']  = ua.str.contains('Tablet',          case=False, na=False).astype(int)
    df['is_desktop'] = ua.str.contains('Desktop/Laptop',  case=False, na=False).astype(int)

    # ── Aggregate to customer level ────────────────────────────────────────────
    agg = df.groupby('customer_id').agg(
        num_logins            = ('login_dt', 'count'),
        days_since_last_login = ('login_dt', lambda x: (reference_date - x.max()).days),
        num_unique_ips        = ('ip', 'nunique'),
        device_type_mobile    = ('is_mobile',  'max'),
        device_type_tablet    = ('is_tablet',  'max'),
        device_type_desktop   = ('is_desktop', 'max'),
    ).reset_index()

    # Login frequency: logins per active day
    login_range = df.groupby('customer_id')['login_dt'].agg(
        first_login='min', last_login='max'
    )
    login_range['active_days'] = (
        (login_range['last_login'] - login_range['first_login']).dt.days + 1
    )
    agg = agg.merge(login_range[['active_days']].reset_index(), on='customer_id', how='left')
    agg['login_frequency_per_day'] = (
        agg['num_logins'] / agg['active_days'].replace(0, 1)
    ).round(4)
    agg = agg.drop(columns=['active_days'])

    logger.info('TRANSFORM LOG: complete  shape=%s', agg.shape)
    return agg

In [12]:
def transform_survey(raw_survey: pd.DataFrame) -> pd.DataFrame:
    """
    Clean and validate the customer survey responses.

    Survey questions (from PerfectPetClub_survey_questions.txt):
    - Q1 reported_income         : income bracket (categorical)
    - Q2 nps_recommend           : likelihood to recommend (0-10)
    - Q3 satisfaction_score      : overall satisfaction (0-10)
    - Q4 online_comparison_freq  : online shopping/price comparison frequency (0-10)

    Data quality performed:
    - Validate NPS / satisfaction / comparison scores are in range 0-10
    - Normalise email column for joining
    - Log and drop rows with invalid scores

    Parameters
    ----------
    raw_survey : pd.DataFrame
        Raw survey DataFrame from extract_survey().

    Returns
    -------
    pd.DataFrame
        Cleaned survey DataFrame with normalised email column.
    """
    logger.info('TRANSFORM SURVEY: starting  raw_shape=%s', raw_survey.shape)

    if raw_survey.empty:
        logger.warning('TRANSFORM SURVEY: empty input – returning empty DataFrame')
        return raw_survey

    df = raw_survey.copy()
    check_quality(df, 'survey')

    # ── Normalise email ────────────────────────────────────────────────────────
    df['email'] = df['email'].str.strip().str.lower()

    # ── Validate 0-10 scale columns ────────────────────────────────────────────
    scale_cols = ['nps_recommend', 'satisfaction_score', 'online_comparison_freq']
    for col in scale_cols:
        if col not in df.columns:
            continue
        out_of_range = df[(df[col] < 0) | (df[col] > 10)]
        if not out_of_range.empty:
            logger.warning('  [WARN] %d rows with %s outside 0-10 – setting to NaN',
                           len(out_of_range), col)
            df.loc[(df[col] < 0) | (df[col] > 10), col] = np.nan
        else:
            logger.info('  [OK] %s values all within 0-10', col)

    # ── Drop duplicate emails (keep last response) ─────────────────────────────
    n_before = len(df)
    df = df.drop_duplicates(subset='email', keep='last')
    logger.info('  Deduplication: %d → %d rows', n_before, len(df))

    logger.info('TRANSFORM SURVEY: complete  shape=%s', df.shape)
    return df

In [13]:
def transform(db_tables: dict,
              raw_crm: pd.DataFrame,
              raw_log: pd.DataFrame,
              raw_survey: pd.DataFrame,
              reference_date: pd.Timestamp) -> pd.DataFrame:
    """
    Orchestrate all transform steps and merge sources into one churn dataset.

    Design decision: LEFT JOIN used throughout so that every customer in the
    database is retained even if they appear in no other source (e.g. never
    contacted CRM, never logged in).  Missing values for such customers are
    filled with 0 for counts and NaN for averages/ratios.

    Parameters
    ----------
    db_tables : dict
        Raw DB tables from extract_db().
    raw_crm : pd.DataFrame
        Raw CRM from extract_crm().
    raw_log : pd.DataFrame
        Raw server log from extract_server_log().
    raw_survey : pd.DataFrame
        Raw survey from extract_survey().
    reference_date : pd.Timestamp
        Date used as 'today' for recency features.

    Returns
    -------
    pd.DataFrame
        Final customer-level churn dataset ready for loading.
    """
    logger.info('TRANSFORM: orchestrating all sources')

    # 1. Transform each source independently
    df_db      = transform_db(db_tables, reference_date)
    df_crm     = transform_crm(raw_crm, reference_date)
    df_log     = transform_server_log(
                     raw_log,
                     db_tables['customer'].set_index('id')['email'],
                     reference_date)
    df_survey  = transform_survey(raw_survey)

    # 2. Join CRM and log features onto customer base
    merged = df_db.copy()
    merged = merged.merge(df_crm,  on='customer_id', how='left')
    merged = merged.merge(df_log,  on='customer_id', how='left')

    # 3. Join survey via email
    if not df_survey.empty:
        merged['email_lower'] = merged['email'].str.strip().str.lower()
        df_survey['email_lower'] = df_survey['email'].str.strip().str.lower()
        survey_cols = ['email_lower', 'nps_recommend',
                       'satisfaction_score', 'online_comparison_freq',
                       'reported_income']
        survey_cols = [c for c in survey_cols if c in df_survey.columns]
        merged = merged.merge(df_survey[survey_cols], on='email_lower', how='left')
        merged = merged.drop(columns=['email_lower'])

    # 4. Fill count NaNs with 0 (customers with no CRM / log activity)
    zero_fill_cols = [
        'num_contacts', 'num_complaints', 'num_queries',
        'num_logins', 'num_unique_ips',
        'uses_e_mail', 'uses_telephone', 'uses_live_chat',
        'device_type_mobile', 'device_type_tablet', 'device_type_desktop',
    ]
    for col in zero_fill_cols:
        if col in merged.columns:
            merged[col] = merged[col].fillna(0).astype(int)

    # 5. Final quality summary
    logger.info('TRANSFORM COMPLETE: shape=%s', merged.shape)
    logger.info('  Churn rate: %.1f%% (%d/%d customers)',
                merged['churn'].mean() * 100,
                merged['churn'].sum(),
                len(merged))
    logger.info('  Columns: %s', list(merged.columns))

    null_summary = merged.isnull().sum()
    null_summary = null_summary[null_summary > 0]
    if not null_summary.empty:
        logger.info('  Remaining NaN columns (expected for unmatched sources):\n%s', null_summary)

    return merged

In [14]:
# ── Run the full transform ─────────────────────────────────────────────────────
churn_df = transform(db_tables, raw_crm, raw_log, raw_survey, REFERENCE_DATE)

print('\n=== Final Churn Dataset ===')
display(churn_df.head())
print(f'\nShape: {churn_df.shape}')
print(f'Churn rate: {churn_df["churn"].mean():.1%}')
print(f'\nData types:\n{churn_df.dtypes}')
print(f'\nMissing values:\n{churn_df.isnull().sum()[churn_df.isnull().sum() > 0]}')

19:44:17 [INFO] TRANSFORM: orchestrating all sources
19:44:17 [INFO] TRANSFORM DB: starting
19:44:17 [INFO] QUALITY CHECK: customer  rows=221  cols=9
19:44:17 [INFO]   [OK] No missing values
19:44:17 [INFO]   [OK] No duplicate rows
19:44:17 [INFO]   [OK] All required columns present
19:44:17 [INFO] QUALITY CHECK: subscription  rows=221  cols=5
19:44:17 [INFO]   [OK] No missing values
19:44:17 [INFO]   [OK] No duplicate rows
19:44:17 [INFO]   [OK] All required columns present
19:44:17 [INFO] QUALITY CHECK: sale  rows=23188  cols=6
19:44:17 [WARNING]   [WARN] Missing: expected_date = 1090 (4.7%)
19:44:17 [WARNING]   [WARN] Missing: delivered_date = 1090 (4.7%)
19:44:17 [INFO]   [OK] No duplicate rows
19:44:17 [INFO]   [OK] All required columns present
19:44:17 [INFO] QUALITY CHECK: payment  rows=23188  cols=4
19:44:17 [INFO]   [OK] No missing values
19:44:17 [INFO]   [OK] No duplicate rows
19:44:17 [INFO]   [OK] All required columns present
19:44:17 [INFO]   [OK] auto_renew values all va


=== Final Churn Dataset ===


,customer_id,email,subscription_id,subscription_type_name,subscription_type_id,subscription_days,auto_renew,churn,num_orders,total_spend_pence,...,uses_e_mail,uses_telephone,uses_live_chat,num_logins,days_since_last_login,num_unique_ips,device_type_mobile,device_type_tablet,device_type_desktop,login_frequency_per_day
0,1,Sargsyan701@somemail.co.uk,1,SuperSaver,3,30,0,1,20,74455,...,1,1,0,48,302,48,0,1,0,4.8000
1,2,Mammadova282@example.com,2,SuperSaver,3,330,1,0,253,1036961,...,1,1,1,1085,0,1085,0,1,0,3.4665
2,3,Sana858@notrealemail.com,3,Premium,1,330,1,0,304,1284692,...,1,1,1,624,0,624,0,1,0,1.9936
3,4,Omar.Farooq4@example.com,4,Premium,1,330,1,0,1055,9577504,...,1,1,1,1540,0,1540,0,1,0,4.9201
4,5,Alejandro357@somemail.co.uk,5,SuperSaver,3,120,0,1,44,179738,...,1,1,1,388,210,388,1,0,0,3.7670



Shape: (221, 31)
Churn rate: 38.9%

Data types:
customer_id                  int64
email                          str
subscription_id              int64
subscription_type_name         str
subscription_type_id         int64
subscription_days            int64
auto_renew                   int64
churn                        int64
num_orders                   int64
total_spend_pence            int64
days_since_last_order        int64
avg_order_value            float64
num_distinct_products        int64
num_reviews                float64
avg_review_rating          float64
avg_delivery_delay_days    float64
num_contacts                 int64
num_complaints               int64
num_queries                  int64
days_since_last_contact    float64
complaint_ratio            float64
uses_e_mail                  int64
uses_telephone               int64
uses_live_chat               int64
num_logins                   int64
days_since_last_login        int64
num_unique_ips               int64
device